# 01 — Item 1 sentences: segmentation, filters, embeddings

**Purpose.** Build the firm side of Stage 4's deterministic rail. Every S&P 500
10-K Item 1 (business description) becomes a table of filtered sentences plus
an aligned embedding matrix — the ground for screening, for the continuous
relevance scores, and for the evidence sentences the classification layer
reads. Per the Stage-4 plan §3, this layer is computation, not inference: no
language model makes any judgment here, and a re-run reproduces the bytes.

**Why sentences, and why these filters.** The plan's rejected-alternatives
table (§3.1): one whole-document embedding dilutes a conglomerate's one
relevant segment; chunk centroids re-introduce the same dilution; LLM
key-sentence selection injects inference into the layer whose purpose is to
be inference-free. Sentence-level units, cleaned by two auditable rules
(length gate, cross-document boilerplate), preserve locality — and use the
**same pinned segmenter** as the hearing-side evidence sentences, so the
later comparison is like-with-like.

**Inputs.** `data/raw/03_sp500_10k_item1/sp500_10k_item1.parquet` — 22,299
filings, 21,966 usable firm-years, 1,061 firms, GVKEY on every row.

**Outputs**, under `interim/relevance/item1/` on scratch:

| File | Contents |
|---|---|
| `sentences.parquet` | one row per kept sentence: accessionNo, cik, gvkey, fyear, sent_idx, sentence |
| `emb_{encoder}.npy` | float16 L2-normalized embeddings, row-aligned to the parquet |
| `boilerplate_report.parquet` | what the cross-document filter removed, for audit |

**Gate (plan §8).** Sentence counts within expectation (~250/doc median),
removal rates reported with threshold sensitivity, zero empty documents
unexplained, encoder name + hidden size logged, embedding rows == parquet rows.

**How to run.** Segmentation is CPU-parallel, embedding needs a GPU — execute
via `jobs/embed_item1.sbatch`. The `ENCODER_NAME` environment variable selects
the encoder; the default is the primary, and the sensitivity re-run sets the
alternative (`relevance.ENCODER_ALT`) without touching this file.

## Setup

The segmenter version is asserted transitively on import (`relevance` →
`stance` → pysbd pin): a wrong environment fails on this cell.

In [1]:
import os
import sys
import time
from collections import Counter
from concurrent.futures import ProcessPoolExecutor

sys.path.insert(0, "../lib")

import numpy as np
import pandas as pd

import relevance as R
from paths import DATA_REPO, HF_CACHE, relevance_run_dir
from stance import PYSBD_VERSION

os.environ["HF_HOME"] = str(HF_CACHE)

ENCODER_NAME = os.environ.get("ENCODER_NAME", R.ENCODER)
RUN_DIR = relevance_run_dir("item1")
SENT_FILE = RUN_DIR / "sentences.parquet"
EMB_FILE = RUN_DIR / f"emb_{ENCODER_NAME.split('/')[-1]}.npy"

print(f"segmenter : pysbd {PYSBD_VERSION}")
print(f"encoder   : {ENCODER_NAME} (pooling: {R.POOLING[ENCODER_NAME]})")
print(f"filters   : min {R.MIN_SENTENCE_CHARS} chars; boilerplate >= "
      f"{R.BOILERPLATE_MIN_FIRMS} firms")
print(f"outputs   : {RUN_DIR}")

segmenter : pysbd 0.3.4
encoder   : BAAI/bge-large-en-v1.5 (pooling: cls)
filters   : min 30 chars; boilerplate >= 20 firms
outputs   : /scratch2/acroh/projects/llm_method/data/interim/relevance/item1


## Step 1 — The usable filings

`usable` and non-empty text only; the 333 unusable and 69 empty rows were
diagnosed at extraction time and carry their reasons in the source table.
Nothing here re-litigates them — this notebook's population is stated, not
chosen.

In [2]:
filings = pd.read_parquet(
    DATA_REPO / "raw" / "03_sp500_10k_item1" / "sp500_10k_item1.parquet",
    columns=["accessionNo", "cik", "gvkey", "conm", "fyear", "filedAt",
             "chars", "usable", "item1"])
filings = filings.loc[filings["usable"] & (filings["chars"] > 0)].reset_index(drop=True)

print(f"filings   : {len(filings):,}")
print(f"firms     : {filings['cik'].nunique():,}")
print(f"fyears    : {filings['fyear'].min()}–{filings['fyear'].max()}")
print(f"chars     : median {filings['chars'].median():,.0f}, "
      f"p90 {filings['chars'].quantile(0.9):,.0f}")

filings   : 21,966
firms     : 1,061
fyears    : 1992–2026
chars     : median 39,828, p90 89,735


## Step 2 — Segment and length-filter

Paragraph blocks first, then the pinned segmenter, then the length gate
(headings and list fragments out). Both the kept and the dropped counts are
measured — a filter whose removal rate is not reported is a silent sample
choice.

In [3]:
def _segment_batch(texts):
    out = []
    for text in texts:
        sentences = R.segment_item1(text)
        out.append(([s for s in sentences if R.keep_sentence(s)], len(sentences)))
    return out

started = time.time()
texts = filings["item1"].tolist()
batches = [texts[i:i + 200] for i in range(0, len(texts), 200)]
with ProcessPoolExecutor() as pool:
    results = [r for batch in pool.map(_segment_batch, batches) for r in batch]

filings["sentences"] = [kept for kept, _ in results]
raw_total = sum(n for _, n in results)
kept_total = int(filings["sentences"].str.len().sum())

n_sent = filings["sentences"].str.len()
print(f"segmented in {(time.time() - started) / 60:.1f} min")
print(f"raw sentences  : {raw_total:,}")
print(f"after length   : {kept_total:,} "
      f"(dropped {1 - kept_total / raw_total:.1%})")
print(f"per filing     : median {n_sent.median():.0f}, "
      f"p90 {n_sent.quantile(0.9):.0f}, max {n_sent.max():,}")
print(f"filings left empty by the gate: {(n_sent == 0).sum()}")

segmented in 10.2 min
raw sentences  : 7,848,199
after length   : 6,331,719 (dropped 19.3%)
per filing     : median 240, p90 532, max 4,266
filings left empty by the gate: 0


## Step 3 — Cross-document boilerplate

A normalized sentence form (digits collapsed, punctuation stripped) that
appears in many *distinct firms'* filings is legal template text — exchange
listings, forward-looking-statement language, employee-count formulae — not
business description. The threshold sensitivity row shows the choice is not
load-bearing, and the top removed forms are printed so the filter's judgment
is inspectable.

In [4]:
key_firms = Counter()
seen_pairs = set()
for cik, sentences in zip(filings["cik"], filings["sentences"]):
    for s in sentences:
        pair = (R.dedup_key(s), cik)
        if pair not in seen_pairs:
            seen_pairs.add(pair)
            key_firms[pair[0]] += 1
del seen_pairs

for threshold in (10, R.BOILERPLATE_MIN_FIRMS, 50):
    bad = {k for k, n in key_firms.items() if n >= threshold}
    removed = sum(1 for ss in filings["sentences"] for s in ss
                  if R.dedup_key(s) in bad)
    marker = " <- adopted" if threshold == R.BOILERPLATE_MIN_FIRMS else ""
    print(f"threshold >= {threshold:>2} firms: {len(bad):>6,} forms, "
          f"{removed / kept_total:.2%} of sentences removed{marker}")

boiler = {k for k, n in key_firms.items() if n >= R.BOILERPLATE_MIN_FIRMS}
examples = {}
for ss in filings["sentences"]:
    for s in ss:
        k = R.dedup_key(s)
        if k in boiler and k not in examples:
            examples[k] = s

top = sorted(boiler, key=lambda k: -key_firms[k])[:8]
print("\nmost widespread boilerplate forms:")
for k in top:
    print(f"  [{key_firms[k]:>4} firms] {examples[k][:90]}")

filings["sentences"] = filings["sentences"].map(
    lambda ss: [s for s in ss if R.dedup_key(s) not in boiler])
final_total = int(filings["sentences"].str.len().sum())
print(f"\nfinal sentences: {final_total:,}")

threshold >= 10 firms:    251 forms, 0.56% of sentences removed


threshold >= 20 firms:     93 forms, 0.39% of sentences removed <- adopted


threshold >= 50 firms:     22 forms, 0.21% of sentences removed



most widespread boilerplate forms:
  [ 263 firms] Executive Officers of the Registrant
  [ 211 firms] Information About our Executive Officers
  [ 188 firms] Diversity, Equity and Inclusion
  [ 144 firms] 3,237 $ 5,615 173.5 27.0 % 2006
  [ 136 firms] 1,169 1,104 2,423 2,598 1,573 1,537
  [ 136 firms] 14,019 3,034 2,711 1,059 547 155
  [ 111 firms] Executive Vice President and Chief Financial Officer
  [ 110 firms] The public may obtain information on the operation of the Public Reference Room by calling



final sentences: 6,306,793


## Step 4 — Spot-check

Eight filings, seeded: what survives the filters should read as substantive
business description, and anything that does not is a filter bug to catch
now.

In [5]:
sample = filings.loc[filings["sentences"].str.len() >= 5].sample(8, random_state=1089)
for row in sample.itertuples():
    print("=" * 78)
    print(f"{row.conm}  fyear {row.fyear}  ({len(row.sentences):,} sentences kept)")
    for s in row.sentences[:4]:
        print(f"  - {s[:100]}")

3M CO  fyear 2005  (161 sentences kept)
  - 3M Company, formerly known as Minnesota Mining and Manufacturing Company, was incorporated in 1929 u
  - The Company’s ticker symbol is MMM.
  - As used herein, the term “3M” or “Company” includes 3M Company and its subsidiaries unless the conte
  - The Company files annual reports, quarterly reports, proxy statements and other documents with the S
HENRY SCHEIN INC  fyear 2016  (374 sentences kept)
  - We believe we are the world’s largest provider of health care products and services primarily to off
  - We serve more than 1 million customers worldwide including dental practitioners and laboratories, an
  - We believe that we have a strong brand identity due to our more than 84 years of experience distribu
  - We are headquartered in Melville, New York, employ more than 21,000 people (of which more than 10,50
STANLEY BLACK & DECKER INC  fyear 2024  (185 sentences kept)
  - Stanley Black & Decker, Inc. ("the Company") was founded in 1843 by F

## Step 5 — Persist the sentence table

One row per sentence. `sent_idx` is the sentence's position after filtering;
the embedding matrix in Step 6 is aligned to this table row-for-row, which is
the only contract downstream code relies on.

In [6]:
table = filings[["accessionNo", "cik", "gvkey", "fyear", "sentences"]].explode(
    "sentences", ignore_index=True).rename(columns={"sentences": "sentence"})
table = table.dropna(subset=["sentence"]).reset_index(drop=True)
table["sent_idx"] = table.groupby("accessionNo").cumcount()

table.to_parquet(SENT_FILE, compression="zstd", index=False)
print(f"wrote {SENT_FILE}")
print(f"  {len(table):,} rows, {SENT_FILE.stat().st_size / 1e6:.0f} MB")

wrote /scratch2/acroh/projects/llm_method/data/interim/relevance/item1/sentences.parquet
  6,306,793 rows, 159 MB


## Step 6 — Embed

The pinned encoder, our pooling, L2-normalized float16 — cosine similarity
downstream is then a dot product, and the full matrix stays ~11 GB. Written
row-aligned to the sentence table; the pairing of the two files is the
artifact.

In [7]:
import torch
from transformers import AutoModel, AutoTokenizer

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)
encoder = AutoModel.from_pretrained(ENCODER_NAME, dtype=torch.float16,
                                    device_map="cuda")
encoder.eval()
print(f"encoder loaded in {time.time() - t0:.0f}s — "
      f"hidden size {encoder.config.hidden_size}")

t0 = time.time()
emb = R.encode(table["sentence"].tolist(), encoder, tokenizer, ENCODER_NAME)
np.save(EMB_FILE, emb)
print(f"embedded {len(emb):,} sentences in {(time.time() - t0) / 60:.0f} min")
print(f"wrote {EMB_FILE}  ({EMB_FILE.stat().st_size / 1e9:.1f} GB)")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

encoder loaded in 2s — hidden size 1024


embedded 6,306,793 sentences in 135 min
wrote /scratch2/acroh/projects/llm_method/data/interim/relevance/item1/emb_bge-large-en-v1.5.npy  (12.9 GB)


## Verification

Alignment and normalization checked on the files as re-read from disk —
these two properties are everything notebook 02 will assume.

In [8]:
back = pd.read_parquet(SENT_FILE)
vecs = np.load(EMB_FILE, mmap_mode="r")

assert len(back) == len(vecs), "embedding rows != sentence rows"
print(f"1) aligned: {len(back):,} sentences == {len(vecs):,} vectors")

norms = np.linalg.norm(vecs[:: max(1, len(vecs) // 1000)].astype(np.float32), axis=1)
assert np.allclose(norms, 1.0, atol=1e-2), "vectors not unit-normalized"
print(f"2) unit norms (sampled): mean {norms.mean():.4f}")

assert not back.duplicated(["accessionNo", "sent_idx"]).any()
print(f"3) key (accessionNo, sent_idx) unique")

per_doc = back.groupby("accessionNo").size()
print(f"4) sentences per filing: median {per_doc.median():.0f}, "
      f"p90 {per_doc.quantile(0.9):.0f} — filings covered: {len(per_doc):,}")

print(f"5) encoder {ENCODER_NAME}, pooling {R.POOLING[ENCODER_NAME]}, "
      f"pysbd {PYSBD_VERSION}")
print("\nDone. Next: 02 (topic centroids + scores — needs the Stage-3 codebook)",
      "and 03 (witness gold — independent, can run any time).")

1) aligned: 6,306,793 sentences == 6,306,793 vectors
2) unit norms (sampled): mean 1.0000


3) key (accessionNo, sent_idx) unique


4) sentences per filing: median 239, p90 530 — filings covered: 21,966
5) encoder BAAI/bge-large-en-v1.5, pooling cls, pysbd 0.3.4

Done. Next: 02 (topic centroids + scores — needs the Stage-3 codebook) and 03 (witness gold — independent, can run any time).
